<a href="https://colab.research.google.com/github/AzadMahmud/dl-lab/blob/main/assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import tensorflow as tf


In [3]:
try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "code":
        ROOT = ROOT.parent

OUT = ROOT / "outputs"
for p in ["part_a", "part_b", "part_c", "part_d"]:
    (OUT / p).mkdir(parents=True, exist_ok=True)

In [4]:
def conv_regular(x: np.ndarray, k: np.ndarray, stride: int, pad: int) -> np.ndarray:
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="constant")
    h, w, c = x_pad.shape
    kh, kw, kc = k.shape
    oh = (h - kh) // stride + 1
    ow = (w - kw) // stride + 1
    y = np.zeros((oh, ow), dtype=np.float32)
    for i in range(oh):
        for j in range(ow):
            patch = x_pad[i * stride : i * stride + kh, j * stride : j * stride + kw, :]
            y[i, j] = np.sum(patch * k)
    return y



In [5]:
def conv_depthwise(x: np.ndarray, k: np.ndarray, stride: int, pad: int) -> np.ndarray:
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="constant")
    h, w, c = x_pad.shape
    kh, kw, kc = k.shape
    oh = (h - kh) // stride + 1
    ow = (w - kw) // stride + 1
    y = np.zeros((oh, ow, c), dtype=np.float32)
    for ch in range(c):
        for i in range(oh):
            for j in range(ow):
                patch = x_pad[i * stride : i * stride + kh, j * stride : j * stride + kw, ch]
                y[i, j, ch] = np.sum(patch * k[:, :, ch])
    return y


In [6]:
def part_a():
    x = np.dstack([
        np.arange(1, 17, dtype=np.float32).reshape(4, 4),
        np.arange(17, 33, dtype=np.float32).reshape(4, 4),
    ])
    k = np.dstack([
        np.array([[1, 0], [0, -1]], dtype=np.float32),
        np.array([[0, 1], [-1, 0]], dtype=np.float32),
    ])

    y1 = conv_regular(x, k, stride=2, pad=0)
    y2 = conv_regular(x, k, stride=2, pad=1)
    y3 = conv_depthwise(x, k, stride=1, pad=0)

    dim = {
        "regular_no_pad_s2": "((4-2+0)/2)+1 = 2 x 2 x 1",
        "regular_pad1_s2": "((4-2+2)/2)+1 = 3 x 3 x 1",
        "depthwise_no_pad_s1": "((4-2+0)/1)+1 = 3 x 3 x 2",
    }
    out = {
        "input_feature_map": x.tolist(),
        "kernel": k.tolist(),
        "dimension_calculations": dim,
        "regular_no_pad_s2": y1.tolist(),
        "regular_pad1_s2": y2.tolist(),
        "depthwise_no_pad_s1": y3.tolist(),
    }
    (OUT / "part_a" / "part_a_results.json").write_text(json.dumps(out, indent=2), encoding="utf-8")

In [7]:
def conv2d_gray(img: np.ndarray, k: np.ndarray) -> np.ndarray:
    kh, kw = k.shape
    p = kh // 2
    x = np.pad(img, ((p, p), (p, p)), mode="edge")
    y = np.zeros_like(img, dtype=np.float32)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            y[i, j] = np.sum(x[i : i + kh, j : j + kw] * k)
    y = (y - y.min()) / (y.max() - y.min() + 1e-8)
    return y


In [8]:
def find_face_image() -> Path:
    candidates = [
        ROOT / "self_image.jpg",
        ROOT.parent / "/content/self_image.jpg",
    ]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError("No face image found. Place self_image.jpg in assignment-4 or use assignment-3/self_image.jpg")

In [9]:
def part_b():
    img_path = find_face_image()
    img = Image.open(img_path).convert("L").resize((256, 256))
    arr = np.asarray(img, dtype=np.float32) / 255.0

    kernels = {
        "edge_sobel_x": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32),
        "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
        "blur_box": (1 / 9.0) * np.ones((3, 3), dtype=np.float32),
    }
    maps = {name: conv2d_gray(arr, k) for name, k in kernels.items()}

    fig, ax = plt.subplots(1, 4, figsize=(14, 4))
    ax[0].imshow(arr, cmap="gray")
    ax[0].set_title("original")
    ax[0].axis("off")
    for i, (name, m) in enumerate(maps.items(), start=1):
        ax[i].imshow(m, cmap="gray")
        ax[i].set_title(name)
        ax[i].axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "part_b" / "kernel_effects.png", dpi=220)
    plt.close(fig)

    (OUT / "part_b" / "kernel_definitions.json").write_text(
        json.dumps({k: v.tolist() for k, v in kernels.items()}, indent=2), encoding="utf-8"
    )


In [10]:
def part_c():
    sep = [
        np.outer([1, 0, -1], [1, 2, 1]),
        np.outer([1, 1, 1], [1, 0, -1]),
        np.outer([1, 2, 1], [1, 1, 1]),
        np.outer([1, -1, 0], [1, 1, 1]),
        np.outer([1, 0, 1], [1, -2, 1]),
        np.outer([2, 1, 0], [0, 1, 2]),
        np.outer([1, 3, 1], [1, 0, -1]),
        np.outer([1, 1, -1], [2, 0, 1]),
        np.outer([0, 1, 2], [1, -1, 1]),
        np.outer([1, -2, 1], [1, 0, 1]),
    ]
    nonsep = [
        np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]]),
        np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]),
        np.array([[1, 0, -1], [0, 0, 0], [-1, 0, 1]]),
        np.array([[2, -1, 0], [-1, 2, -1], [0, -1, 2]]),
        np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
        np.array([[1, 2, 1], [2, -12, 2], [1, 2, 1]]),
        np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 0]]),
        np.array([[0, 2, -1], [1, -3, 1], [-1, 2, 0]]),
        np.array([[3, -1, 0], [-1, -1, -1], [0, -1, 3]]),
        np.array([[1, -1, 1], [-1, 0, -1], [1, -1, 0]]),
    ]

    def rec(mats, tag):
        return [{"name": f"{tag}_{i+1}", "matrix": m.tolist(), "rank": int(np.linalg.matrix_rank(m))} for i, m in enumerate(mats)]

    data = {"separable": rec(sep, "sep"), "non_separable": rec(nonsep, "nonsep")}
    (OUT / "part_c" / "separability_kernels.json").write_text(json.dumps(data, indent=2), encoding="utf-8")

In [11]:
def part_d():
    model = tf.keras.applications.VGG16(weights="imagenet", include_top=False)
    w = model.get_layer("block1_conv1").get_weights()[0]  # (3,3,3,64)
    k = w.mean(axis=2)  # (3,3,64)
    sobx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    soby = sobx.T

    rows = []
    for i in range(k.shape[-1]):
        f = k[:, :, i]
        f0 = f - f.mean()
        nx = float(np.dot(f0.ravel(), sobx.ravel()) / ((np.linalg.norm(f0) * np.linalg.norm(sobx)) + 1e-8))
        ny = float(np.dot(f0.ravel(), soby.ravel()) / ((np.linalg.norm(f0) * np.linalg.norm(soby)) + 1e-8))
        rows.append((i, nx, ny, max(abs(nx), abs(ny))))

    rows = sorted(rows, key=lambda x: x[3], reverse=True)
    top = rows[:8]
    np.savetxt(OUT / "part_d" / "sobel_similarity_top8.csv", np.array(top), delimiter=",", header="filter,cos_sobel_x,cos_sobel_y,max_abs", comments="")

    fig, ax = plt.subplots(2, 4, figsize=(8, 4))
    for a, (idx, nx, ny, _) in zip(ax.flat, top):
        f = k[:, :, int(idx)]
        f = (f - f.min()) / (f.max() - f.min() + 1e-8)
        a.imshow(f, cmap="seismic", vmin=0, vmax=1)
        a.set_title(f"f{int(idx)}\n({nx:.2f},{ny:.2f})", fontsize=8)
        a.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "part_d" / "vgg16_sobel_like_filters.png", dpi=220)
    plt.close(fig)


In [12]:
if __name__ == "__main__":
    part_a()
    part_b()
    part_c()
    part_d()
    print("Done. Outputs at", OUT)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Done. Outputs at /content/outputs
